# 06 - DVAE on IMU (disentangled sex / weight latents)

Debiasing VAE following Rahman et al. 2025: two parallel encoders produce a sex-specific latent (`z_sex`) and a sex-agnostic latent (`z_agn`). A weight regressor reads `z_agn`; a sex classifier reads `z_sex`. Gradient-reversal adversarial heads push the encoders to keep sex out of `z_agn` and weight out of `z_sex`. A single decoder reconstructs from `concat(z_agn, z_sex)`.

At inference only `z_agn` is used (sex encoder is discarded), so predictions are based on sex-invariant motion features. Inputs, split, and preprocessing are identical to notebook 04.

Outputs to `VAE/Results/dvae_imu/`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    mean_absolute_error, mean_squared_error, confusion_matrix
)

ROOT = Path(r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project")
TENSOR_PATH = ROOT / "VAE" / "Tensors" / "lifts_imu.npz"
OUT_DIR = ROOT / "VAE" / "Results" / "dvae_imu"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Latent sizes: z_agn carries the weight signal, z_sex carries the (small) binary sex signal
LATENT_AGN = 32
LATENT_SEX = 8
BETA = 0.1                # KL weight, same as VAE
LAMBDA_MAIN = 10.0        # main heads: weight-from-agn (MSE) + sex-from-sex (CE)
LAMBDA_ADV = 1.0          # adversarial heads via GRL
GRL_LAMBDA = 1.0          # gradient reversal strength
BATCH_SIZE = 64
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-5
LOAD_BINS = np.array([2.3, 4.5, 6.8, 9.1, 11.3])

Device: cpu


In [2]:
# ---- load + drop NaN lifts ----
data = np.load(TENSOR_PATH, allow_pickle=True)
X = data["X"].astype(np.float32)
participant = data["participant"]
sex = data["sex"]
box_mass = data["box_mass"].astype(np.float32)
channel_names = data["channel_names"]

nan_lift = np.isnan(X).any(axis=(1, 2))
print(f"Dropping {int(nan_lift.sum())} / {len(X)} lifts with NaN")
if nan_lift.any():
    u, c = np.unique(participant[nan_lift], return_counts=True)
    print("  by subject:", dict(zip(u.tolist(), c.tolist())))
keep = ~nan_lift
X = X[keep]; participant = participant[keep]; sex = sex[keep]; box_mass = box_mass[keep]

def mass_to_class(m): return int(np.argmin(np.abs(LOAD_BINS - m)))
y_cls = np.array([mass_to_class(m) for m in box_mass], dtype=np.int64)

# sex as 0/1
sex_labels = np.array(["Female", "Male"])
sex_to_idx = {s: i for i, s in enumerate(sex_labels)}
y_sex = np.array([sex_to_idx[s] for s in sex], dtype=np.int64)
print("Sex dist:", dict(zip(*np.unique(y_sex, return_counts=True))))
print("X shape:", X.shape, "subjects:", len(np.unique(participant)))

Dropping 261 / 4800 lifts with NaN
  by subject: {'P10': 12, 'P16': 9, 'P30': 120, 'P38': 120}
Sex dist: {0: 2379, 1: 2160}
X shape: (4539, 400, 96) subjects: 38


In [3]:
# ---- subject-grouped 80/20 split, sex-stratified ----
subjects = np.unique(participant)
subj_sex = {p: sex[participant == p][0] for p in subjects}
rng = np.random.default_rng(SEED)
test_subjects = []
for s_label in np.unique(list(subj_sex.values())):
    subs_s = np.array([p for p, s in subj_sex.items() if s == s_label])
    rng.shuffle(subs_s)
    n_test = max(1, int(round(0.2 * len(subs_s))))
    test_subjects.extend(subs_s[:n_test].tolist())
test_subjects = set(test_subjects)
train_subjects = set(subjects) - test_subjects
train_mask = np.array([p in train_subjects for p in participant])
test_mask = ~train_mask
print(f"Train subjects: {len(train_subjects)} lifts: {train_mask.sum()}  |  Test subjects: {len(test_subjects)} lifts: {test_mask.sum()}")

Train subjects: 30 lifts: 3579  |  Test subjects: 8 lifts: 960


In [4]:
# ---- channel z-score and target normalization (fit on train) ----
X_train_raw = X[train_mask]
ch_mean = X_train_raw.reshape(-1, X.shape[-1]).mean(axis=0)
ch_std = X_train_raw.reshape(-1, X.shape[-1]).std(axis=0) + 1e-6
Xn = (X - ch_mean) / ch_std

y_mean = float(box_mass[train_mask].mean())
y_std = float(box_mass[train_mask].std() + 1e-6)
y_norm = (box_mass - y_mean) / y_std

X_train = torch.from_numpy(Xn[train_mask]).transpose(1, 2)
X_test = torch.from_numpy(Xn[test_mask]).transpose(1, 2)
yw_train = torch.from_numpy(y_norm[train_mask]).float()
yw_test = torch.from_numpy(y_norm[test_mask]).float()
ys_train = torch.from_numpy(y_sex[train_mask])
ys_test = torch.from_numpy(y_sex[test_mask])
cls_train = y_cls[train_mask]

# class-balanced sampler (on weight class) to keep both extremes in every batch
class_counts = np.bincount(cls_train, minlength=len(LOAD_BINS))
sample_w = 1.0 / class_counts[cls_train]
sampler = WeightedRandomSampler(weights=torch.from_numpy(sample_w).double(),
                                num_samples=len(sample_w), replacement=True)
train_loader = DataLoader(TensorDataset(X_train, yw_train, ys_train),
                          batch_size=BATCH_SIZE, sampler=sampler)
test_loader = DataLoader(TensorDataset(X_test, yw_test, ys_test),
                         batch_size=BATCH_SIZE, shuffle=False)
print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: torch.Size([3579, 96, 400]) X_test: torch.Size([960, 96, 400])


In [5]:
# ---- gradient reversal layer ----
class GRLFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None
def grl(x, lambda_=1.0):
    return GRLFn.apply(x, lambda_)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(in_c, out_c, k, s, p), nn.BatchNorm1d(out_c), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class DeconvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p, op):
        super().__init__()
        self.net = nn.Sequential(nn.ConvTranspose1d(in_c, out_c, k, s, p, output_padding=op), nn.BatchNorm1d(out_c), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class Encoder(nn.Module):
    def __init__(self, n_channels, seq_len, latent_dim, base=64):
        super().__init__()
        self.enc = nn.Sequential(
            ConvBlock(n_channels, base, 7, 2, 3),
            ConvBlock(base, base*2, 5, 2, 2),
            ConvBlock(base*2, base*4, 3, 2, 1),
        )
        self.out_len = seq_len // 8
        self.flat_dim = base*4 * self.out_len
        self.fc_mu = nn.Linear(self.flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim, latent_dim)
    def forward(self, x):
        h = self.enc(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

class Decoder(nn.Module):
    def __init__(self, n_channels, seq_len, latent_total, base=64):
        super().__init__()
        self.out_len = seq_len // 8
        self.fc = nn.Linear(latent_total, base*4 * self.out_len)
        self.dec = nn.Sequential(
            DeconvBlock(base*4, base*2, 3, 2, 1, 1),
            DeconvBlock(base*2, base, 5, 2, 2, 1),
            nn.ConvTranspose1d(base, n_channels, 7, 2, 3, output_padding=1),
        )
        self.base = base
    def forward(self, z):
        return self.dec(self.fc(z).view(-1, self.base*4, self.out_len))

class DVAE(nn.Module):
    def __init__(self, n_channels, seq_len=400, latent_agn=32, latent_sex=8, n_load_classes=5):
        super().__init__()
        self.enc_agn = Encoder(n_channels, seq_len, latent_agn, base=64)
        self.enc_sex = Encoder(n_channels, seq_len, latent_sex, base=32)
        self.dec = Decoder(n_channels, seq_len, latent_agn + latent_sex, base=64)
        # main heads
        self.reg_main = nn.Sequential(nn.Linear(latent_agn, 64), nn.ReLU(inplace=True), nn.Linear(64, 1))
        self.cls_main = nn.Sequential(nn.Linear(latent_sex, 32), nn.ReLU(inplace=True), nn.Linear(32, 2))
        # adversarial heads (see sex from z_agn and weight from z_sex)
        self.cls_adv = nn.Sequential(nn.Linear(latent_agn, 32), nn.ReLU(inplace=True), nn.Linear(32, 2))
        self.reg_adv = nn.Sequential(nn.Linear(latent_sex, 32), nn.ReLU(inplace=True), nn.Linear(32, 1))

    def reparam(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

    def forward(self, x, grl_lambda=1.0):
        mu_a, lv_a = self.enc_agn(x)
        mu_s, lv_s = self.enc_sex(x)
        z_a = self.reparam(mu_a, lv_a)
        z_s = self.reparam(mu_s, lv_s)
        x_rec = self.dec(torch.cat([z_a, z_s], dim=1))
        w_main = self.reg_main(z_a).squeeze(-1)
        s_main = self.cls_main(z_s)
        # adversarial paths: GRL on the encoder side; adv head itself trains normally
        s_adv = self.cls_adv(grl(z_a, grl_lambda))
        w_adv = self.reg_adv(grl(z_s, grl_lambda)).squeeze(-1)
        return x_rec, (mu_a, lv_a, mu_s, lv_s), (w_main, s_main, s_adv, w_adv)

model = DVAE(n_channels=X.shape[-1], seq_len=X.shape[1],
             latent_agn=LATENT_AGN, latent_sex=LATENT_SEX).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 1,874,006


In [6]:
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
history = []

def kl(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr = {"rec": 0.0, "kl": 0.0, "reg": 0.0, "cls": 0.0, "adv_cls": 0.0, "adv_reg": 0.0,
          "n": 0, "w_abs": 0.0, "s_correct": 0}
    for xb, ywb, ysb in train_loader:
        xb = xb.to(device); ywb = ywb.to(device); ysb = ysb.to(device)
        opt.zero_grad()
        x_rec, (mu_a, lv_a, mu_s, lv_s), (w_main, s_main, s_adv, w_adv) = model(xb, grl_lambda=GRL_LAMBDA)
        L_rec = F.mse_loss(x_rec, xb)
        L_kl = kl(mu_a, lv_a) + kl(mu_s, lv_s)
        L_reg = F.mse_loss(w_main, ywb)
        L_cls = F.cross_entropy(s_main, ysb)
        L_adv_cls = F.cross_entropy(s_adv, ysb)        # trains adv head; reversed gradient reaches enc_agn
        L_adv_reg = F.mse_loss(w_adv, ywb)             # trains adv head; reversed gradient reaches enc_sex
        L = L_rec + BETA*L_kl + LAMBDA_MAIN*(L_reg + L_cls) + LAMBDA_ADV*(L_adv_cls + L_adv_reg)
        L.backward(); opt.step()
        bs = xb.size(0)
        tr["rec"] += L_rec.item()*bs; tr["kl"] += L_kl.item()*bs
        tr["reg"] += L_reg.item()*bs; tr["cls"] += L_cls.item()*bs
        tr["adv_cls"] += L_adv_cls.item()*bs; tr["adv_reg"] += L_adv_reg.item()*bs
        tr["w_abs"] += (w_main - ywb).abs().sum().item()
        tr["s_correct"] += (s_main.argmax(1) == ysb).sum().item()
        tr["n"] += bs

    model.eval()
    te = {"rec": 0.0, "kl": 0.0, "reg": 0.0, "cls": 0.0, "adv_cls": 0.0, "adv_reg": 0.0,
          "n": 0, "w_abs": 0.0, "s_correct": 0, "s_from_agn_correct": 0}
    with torch.no_grad():
        for xb, ywb, ysb in test_loader:
            xb = xb.to(device); ywb = ywb.to(device); ysb = ysb.to(device)
            x_rec, (mu_a, lv_a, mu_s, lv_s), (w_main, s_main, s_adv, w_adv) = model(xb, grl_lambda=GRL_LAMBDA)
            L_rec = F.mse_loss(x_rec, xb)
            L_kl = kl(mu_a, lv_a) + kl(mu_s, lv_s)
            L_reg = F.mse_loss(w_main, ywb)
            L_cls = F.cross_entropy(s_main, ysb)
            L_adv_cls = F.cross_entropy(s_adv, ysb)
            L_adv_reg = F.mse_loss(w_adv, ywb)
            bs = xb.size(0)
            te["rec"] += L_rec.item()*bs; te["kl"] += L_kl.item()*bs
            te["reg"] += L_reg.item()*bs; te["cls"] += L_cls.item()*bs
            te["adv_cls"] += L_adv_cls.item()*bs; te["adv_reg"] += L_adv_reg.item()*bs
            te["w_abs"] += (w_main - ywb).abs().sum().item()
            te["s_correct"] += (s_main.argmax(1) == ysb).sum().item()
            te["s_from_agn_correct"] += (s_adv.argmax(1) == ysb).sum().item()
            te["n"] += bs

    row = {"epoch": epoch,
           "tr_rec": tr["rec"]/tr["n"], "tr_kl": tr["kl"]/tr["n"],
           "tr_reg": tr["reg"]/tr["n"], "tr_cls": tr["cls"]/tr["n"],
           "tr_advcls": tr["adv_cls"]/tr["n"], "tr_advreg": tr["adv_reg"]/tr["n"],
           "tr_sex_acc": tr["s_correct"]/tr["n"],
           "te_rec": te["rec"]/te["n"], "te_kl": te["kl"]/te["n"],
           "te_reg": te["reg"]/te["n"], "te_cls": te["cls"]/te["n"],
           "te_advcls": te["adv_cls"]/te["n"], "te_advreg": te["adv_reg"]/te["n"],
           "te_sex_main_acc": te["s_correct"]/te["n"],
           "te_sex_from_agn_acc": te["s_from_agn_correct"]/te["n"]}
    history.append(row)
    if epoch % 5 == 0 or epoch == 1:
        print(f"ep{epoch:3d} | rec {row['tr_rec']:.3f} kl {row['tr_kl']:.3f} reg {row['tr_reg']:.3f} cls {row['tr_cls']:.3f} "
              f"adv_c {row['tr_advcls']:.3f} adv_r {row['tr_advreg']:.3f} | "
              f"te_reg {row['te_reg']:.3f} sex_main {row['te_sex_main_acc']:.3f} sex_from_agn {row['te_sex_from_agn_acc']:.3f}")

pd.DataFrame(history).to_csv(OUT_DIR / "history.csv", index=False)

ep  1 | rec 0.974 kl 683.834 reg 1.018 cls 0.403 adv_c 2.608 adv_r 53.011 | te_reg 0.915 sex_main 0.517 sex_from_agn 0.547
ep  5 | rec 0.721 kl 313.939 reg 0.356 cls 0.236 adv_c 0.865 adv_r 31.268 | te_reg 0.960 sex_main 0.730 sex_from_agn 0.495
ep 10 | rec 0.665 kl 13.524 reg 0.183 cls 0.144 adv_c 0.708 adv_r 1.298 | te_reg 0.771 sex_main 0.811 sex_from_agn 0.468
ep 15 | rec 0.620 kl 8.932 reg 0.105 cls 0.122 adv_c 0.721 adv_r 1.153 | te_reg 0.835 sex_main 0.803 sex_from_agn 0.401
ep 20 | rec 0.589 kl 7.468 reg 0.064 cls 0.101 adv_c 0.710 adv_r 1.083 | te_reg 0.766 sex_main 0.795 sex_from_agn 0.398
ep 25 | rec 0.546 kl 7.412 reg 0.045 cls 0.084 adv_c 0.700 adv_r 1.083 | te_reg 0.745 sex_main 0.750 sex_from_agn 0.461
ep 30 | rec 0.525 kl 6.821 reg 0.035 cls 0.090 adv_c 0.698 adv_r 1.099 | te_reg 0.712 sex_main 0.803 sex_from_agn 0.473
ep 35 | rec 0.519 kl 8.047 reg 0.041 cls 0.079 adv_c 0.696 adv_r 1.078 | te_reg 0.740 sex_main 0.790 sex_from_agn 0.495
ep 40 | rec 0.508 kl 8.722 reg 0.

In [7]:
# ---- inference: predict weight from z_agn (mu_a) only, as in paper ----
model.eval()
all_mu_a, all_mu_s, all_pred_norm, all_sex_main, all_sex_adv = [], [], [], [], []
with torch.no_grad():
    for xb, ywb, ysb in test_loader:
        xb = xb.to(device)
        mu_a, _ = model.enc_agn(xb)
        mu_s, _ = model.enc_sex(xb)
        w_pred = model.reg_main(mu_a).squeeze(-1)
        s_main = model.cls_main(mu_s).argmax(1)
        s_adv = model.cls_adv(mu_a).argmax(1)   # should be near chance if disentanglement worked
        all_mu_a.append(mu_a.cpu().numpy())
        all_mu_s.append(mu_s.cpu().numpy())
        all_pred_norm.append(w_pred.cpu().numpy())
        all_sex_main.append(s_main.cpu().numpy())
        all_sex_adv.append(s_adv.cpu().numpy())

mu_a_test = np.concatenate(all_mu_a)
mu_s_test = np.concatenate(all_mu_s)
pred_norm = np.concatenate(all_pred_norm)
sex_main_pred = np.concatenate(all_sex_main)
sex_adv_pred = np.concatenate(all_sex_adv)

mass_pred = pred_norm * y_std + y_mean
mass_true = box_mass[test_mask]
y_true_cls = np.array([mass_to_class(m) for m in mass_true])
y_pred_cls = np.array([mass_to_class(m) for m in mass_pred])

mae = mean_absolute_error(mass_true, mass_pred)
rmse = np.sqrt(mean_squared_error(mass_true, mass_pred))
acc = accuracy_score(y_true_cls, y_pred_cls)
bal_acc = balanced_accuracy_score(y_true_cls, y_pred_cls)
sex_main_acc = accuracy_score(y_sex[test_mask], sex_main_pred)
sex_adv_acc = accuracy_score(y_sex[test_mask], sex_adv_pred)

print(f"Weight  MAE={mae:.3f} kg  RMSE={rmse:.3f} kg  nearest-class acc={acc:.3f}  bal_acc={bal_acc:.3f}")
print(f"Sex classifier (from z_sex): acc={sex_main_acc:.3f}  (should be high)")
print(f"Sex classifier (from z_agn): acc={sex_adv_acc:.3f}  (should be near 0.5 if disentangled)")
print("\nMean predicted kg by true class:")
for k, m in enumerate(LOAD_BINS):
    sel = y_true_cls == k
    if sel.any():
        print(f"  true={m:5.2f} kg  n={int(sel.sum()):3d}  pred_mean={mass_pred[sel].mean():.3f}  pred_std={mass_pred[sel].std():.3f}")
print(f"\npred/true std ratio: {mass_pred.std()/mass_true.std():.3f}")

Weight  MAE=2.184 kg  RMSE=2.729 kg  nearest-class acc=0.323  bal_acc=0.323
Sex classifier (from z_sex): acc=0.774  (should be high)
Sex classifier (from z_agn): acc=0.510  (should be near 0.5 if disentangled)

Mean predicted kg by true class:
  true= 2.30 kg  n=192  pred_mean=5.121  pred_std=1.964
  true= 4.50 kg  n=192  pred_mean=6.394  pred_std=2.151
  true= 6.80 kg  n=192  pred_mean=7.378  pred_std=1.877
  true= 9.10 kg  n=192  pred_mean=8.041  pred_std=1.800
  true=11.30 kg  n=192  pred_mean=8.746  pred_std=1.575

pred/true std ratio: 0.711


In [8]:
# ---- fairness by sex + Rahman-style SP/PRD/NRD + save ----
test_sex = sex[test_mask]
signed_err = mass_pred - mass_true
abs_err = np.abs(signed_err)

fair_rows = []
for s_label in np.unique(test_sex):
    m = test_sex == s_label
    fair_rows.append({
        "sex": s_label, "n": int(m.sum()),
        "mae_kg": float(abs_err[m].mean()),
        "rmse_kg": float(np.sqrt((signed_err[m] ** 2).mean())),
        "mean_signed_err_kg": float(signed_err[m].mean()),
        "over_rate": float((signed_err[m] > 0).mean()),
        "under_rate": float((signed_err[m] < 0).mean()),
        "nearest_acc": accuracy_score(y_true_cls[m], y_pred_cls[m]),
    })
fair_df = pd.DataFrame(fair_rows)
print(fair_df.to_string(index=False))

# Rahman et al 2025 fairness metrics (on the test set)
f_mask = test_sex == "Female"; m_mask = test_sex == "Male"
SP = float(mass_pred[f_mask].mean() - mass_pred[m_mask].mean())                      # closer to 0 = fairer
PRD = float(abs(np.maximum(0, mass_true[f_mask] - mass_pred[f_mask]).mean()
                - np.maximum(0, mass_true[m_mask] - mass_pred[m_mask]).mean()))      # under-estimation parity
NRD = float(abs(np.minimum(0, mass_true[f_mask] - mass_pred[f_mask]).mean()
                - np.minimum(0, mass_true[m_mask] - mass_pred[m_mask]).mean()))      # over-estimation parity
print(f"\nSP (F-M mean pred diff)      = {SP:+.3f} kg   (0 = ideal)")
print(f"PRD (underestimation parity) = {PRD:.3f} kg   (0 = ideal)")
print(f"NRD (overestimation parity)  = {NRD:.3f} kg   (0 = ideal)")

torch.save({"model_state": model.state_dict(), "ch_mean": ch_mean, "ch_std": ch_std,
            "y_mean": y_mean, "y_std": y_std,
            "latent_agn": LATENT_AGN, "latent_sex": LATENT_SEX,
            "seq_len": X.shape[1], "n_channels": X.shape[-1],
            "channel_names": list(map(str, channel_names))}, OUT_DIR / "dvae_imu.pt")
np.savez(OUT_DIR / "test_outputs.npz",
         mu_agn=mu_a_test, mu_sex=mu_s_test,
         mass_true=mass_true, mass_pred=mass_pred,
         y_true_cls=y_true_cls, y_pred_cls=y_pred_cls,
         sex_main_pred=sex_main_pred, sex_adv_pred=sex_adv_pred,
         participant=participant[test_mask], sex=test_sex)
with open(OUT_DIR / "summary.json", "w") as f:
    json.dump({"mae_kg": float(mae), "rmse_kg": float(rmse),
               "nearest_acc": float(acc), "nearest_bal_acc": float(bal_acc),
               "sex_main_acc": float(sex_main_acc), "sex_from_agn_acc": float(sex_adv_acc),
               "SP": SP, "PRD": PRD, "NRD": NRD,
               "confusion_matrix": confusion_matrix(y_true_cls, y_pred_cls).tolist(),
               "fairness": fair_rows,
               "config": {"latent_agn": LATENT_AGN, "latent_sex": LATENT_SEX,
                          "beta": BETA, "lambda_main": LAMBDA_MAIN,
                          "lambda_adv": LAMBDA_ADV, "grl_lambda": GRL_LAMBDA,
                          "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LR, "seed": SEED}}, f, indent=2)
fair_df.to_csv(OUT_DIR / "fairness.csv", index=False)
print("Saved to:", OUT_DIR)

   sex   n   mae_kg  rmse_kg  mean_signed_err_kg  over_rate  under_rate  nearest_acc
Female 480 1.950756 2.446275           -0.029905   0.495833    0.504167     0.362500
  Male 480 2.416695 2.985898            0.701692   0.568750    0.431250     0.283333

SP (F-M mean pred diff)      = -0.732 kg   (0 = ideal)
PRD (underestimation parity) = 0.133 kg   (0 = ideal)
NRD (overestimation parity)  = 0.599 kg   (0 = ideal)
Saved to: C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\VAE\Results\dvae_imu
